In [40]:
import torch
from torch_geometric.datasets import WikipediaNetwork, TUDataset, Planetoid, Coauthor, CitationFull, ZINC, QM9, KarateClub
import numpy as np
from torch_geometric.utils import subgraph, to_scipy_sparse_matrix, degree
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [41]:
def modularity(edge_index, communities, degrees):
    row, col = edge_index
    m = degrees.sum()
    modularity_score = 0.0
    for c in torch.unique(communities):
        community_nodes = (communities == c).nonzero(as_tuple=True)[0]
        k_i = degrees[community_nodes].sum()
        e_ii = 0.0
        for node in community_nodes:
            neighbors = col[row == node]
            e_ii += (communities[neighbors] == c).sum().item()
        modularity_score += (e_ii / m) - (k_i / m) ** 2

    return modularity_score

def louvain(data, max_iter=10):
    edge_index = data.edge_index
    num_nodes = data.num_nodes
    degrees = degree(edge_index[0], num_nodes, dtype=torch.float).to(device)
    communities = torch.arange(num_nodes).to(device)

    prev_modularity = -1.0
    for _ in tqdm(range(max_iter)):
        print("Iteration: ",_)
        for node in tqdm(range(num_nodes)):
            neighbors = edge_index[1, edge_index[0] == node].to(device)
            best_community = communities[node]
            best_modularity = prev_modularity

            for community in torch.unique(communities[neighbors]):
                temp_communities = communities.clone()
                temp_communities[node] = community
                new_modularity = modularity(edge_index, temp_communities, degrees)

                if new_modularity > best_modularity:
                    best_modularity = new_modularity
                    best_community = community

            communities[node] = best_community

        current_modularity = modularity(edge_index, communities, degrees)
        if abs(current_modularity - prev_modularity) < 1e-5:
            break
        prev_modularity = current_modularity

    mapping = {}
    for i, c in enumerate(communities):
        if int(c) not in mapping.keys():
            mapping[int(c)] = []
        mapping[int(c)].append(i)
    return mapping

def merge_communities(data, mapping, k):
    sorted_mapping = sorted(mapping.items(), key=lambda x: len(x[1]), reverse=True)
    new_nodes = torch.tensor([], dtype=torch.long).to(device)
    for i in range(len(sorted_mapping)):
        if len(new_nodes) + len(sorted_mapping[i][1]) <= k:
            new_nodes = torch.cat((new_nodes, torch.tensor(sorted_mapping[i][1], dtype=torch.long).to(device)))
            if len(new_nodes) == k:
                break
    new_data = data.subgraph(new_nodes)
    return new_data

In [56]:
dataset = Planetoid(root='./dataset', name='Cora')
data = dataset[0].to(device)


In [51]:
from time import time
start = time()
mapping = louvain(data)
new_data = merge_communities(data, mapping, 5)
print("Time taken: ", time()-start)

  0%|          | 0/10 [00:00<?, ?it/s]

Iteration:  0


 10%|█         | 1/10 [00:01<00:11,  1.27s/it]

Iteration:  1


 20%|██        | 2/10 [00:01<00:07,  1.07it/s]

...

Iteration:  3


 40%|████      | 4/10 [00:02<00:03,  1.59it/s]

Iteration:  4


 40%|████      | 4/10 [00:03<00:05,  1.17it/s]

Time taken:  3.426990032196045


In [45]:
import igraph as ig
import leidenalg

In [60]:
g = ig.Graph(n=data.num_nodes, edges=data.edge_index.t().tolist())

In [61]:
from time import time
start = time()
part = leidenalg.find_partition(g, leidenalg.ModularityVertexPartition)
mapping = {}
for i, c in enumerate(part.membership):
    if int(c) not in mapping.keys():
        mapping[int(c)] = []
    mapping[int(c)].append(i)
new_data_2 = merge_communities(data, mapping, 100)
print("Time taken: ", time() - start)


Time taken:  0.14850306510925293


In [62]:
new_data_2.edge_index, new_data.edge_index

(tensor([[ 0,  0,  1,  1,  1,  1,  2,  2,  2,  3,  3,  4,  4,  4,  4,  4,  5,  6,
           7,  7,  7,  7,  8,  8,  9,  9,  9,  9,  9, 10, 10, 10, 10, 10, 10, 10,
          10, 10, 10, 11, 11, 12, 12, 12, 13, 13, 13, 14, 14, 14, 15, 16, 16, 17,
          17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 18, 19, 19, 19,
          20, 20, 20, 20, 21, 21, 21, 21, 21, 22, 22, 22, 23, 24, 24, 24, 25, 25,
          26, 26, 26, 27, 27, 27, 28, 28, 29, 29, 29, 30, 30, 30, 31, 31, 31, 32,
          32, 33, 33, 33, 34, 35, 35, 35, 35, 35, 35, 35, 36, 37, 37, 37, 38, 38,
          38, 38, 38, 38, 39, 39, 39, 39, 39, 39, 39, 39, 40, 40, 41, 41, 41, 41,
          41, 42, 43, 44, 44, 45, 45, 45, 46, 47, 48, 48, 48, 48, 49, 49, 49, 49,
          49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 50, 50, 50, 50, 51, 52, 52,
          52, 52, 53, 53, 53, 53, 53, 53, 54, 54, 54, 55, 55, 55, 56, 56, 56, 57,
          57, 58, 58, 58, 59, 59, 59, 59, 60, 60, 60, 60, 60, 60, 61, 61, 61, 61,
          61, 62